In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score

In [2]:
pip install optuna

Note: you may need to restart the kernel to use updated packages.


In [3]:
train_df = pd.read_csv(r"C:\Users\ruchi\Desktop\SentinelNet-NIDS\data\processed\NSL_KDD_Train_Clean.csv")
test_df = pd.read_csv(r"C:\Users\ruchi\Desktop\SentinelNet-NIDS\data\processed\NSL_KDD_Test_Clean.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (125973, 42)
Test shape: (22544, 42)


In [4]:
X_train = train_df.iloc[:, :-2]
y_train = train_df.iloc[:, -2]

X_test = test_df.iloc[:, :-2]
y_test = test_df.iloc[:, -2]

# Label encoding
y_train = y_train.astype(str).apply(lambda x: 0 if "normal" in x.lower() else 1)
y_test = y_test.astype(str).apply(lambda x: 0 if "normal" in x.lower() else 1)

print("y_train distribution:\n", y_train.value_counts())
print("y_test distribution:\n", y_test.value_counts())

y_train distribution:
 class
0    67343
1    58630
Name: count, dtype: int64
y_test distribution:
 class
1    12833
0     9711
Name: count, dtype: int64


In [5]:
# One-hot encoding
X_train = pd.get_dummies(X_train)
X_test = pd.get_dummies(X_test)

# ✅ 'inner' join — yahi fix hai, pehle 'left' tha jo galat tha
X_train, X_test = X_train.align(X_test, join='inner', axis=1, fill_value=0)

print("X_train shape after alignment:", X_train.shape)
print("X_test shape after alignment:", X_test.shape)

X_train shape after alignment: (125973, 115)
X_test shape after alignment: (22544, 115)


In [6]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [7]:
# Class imbalance fix
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
ratio = neg / pos

model = XGBClassifier(
    n_estimators=500,        # ← 300 se badhakar 500
    max_depth=8,             # ← 6 se badhakar 8
    learning_rate=0.03,      # ← aur slow learning
    subsample=0.85,
    colsample_bytree=0.85,
    scale_pos_weight=ratio,
    reg_alpha=0.3,
    reg_lambda=1.0,
    min_child_weight=3,      # ← naya: overfitting rokta hai
    gamma=0.1,               # ← naya: splitting control
    random_state=42,
    eval_metric='logloss'
)

model.fit(X_train, y_train)
print("✅ Model trained!")

✅ Model trained!


In [8]:
from sklearn.metrics import f1_score

# Probability nikalo
y_prob = model.predict_proba(X_test)[:, 1]

# Best threshold dhundo
best_thresh = 0.5
best_f1 = 0

for thresh in [x/100 for x in range(30, 70)]:
    y_temp = (y_prob >= thresh).astype(int)
    f1 = f1_score(y_test, y_temp)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = thresh

print(f"Best Threshold: {best_thresh}")
print(f"Best F1 Score: {best_f1:.4f}")

Best Threshold: 0.3
Best F1 Score: 0.7932


In [9]:
# Best threshold se predict karo
y_pred_final = (y_prob >= best_thresh).astype(int)

print("✅ Final Test Accuracy:", accuracy_score(y_test, y_pred_final))
print()
print(classification_report(y_test, y_pred_final))

✅ Final Test Accuracy: 0.8007008516678495

              precision    recall  f1-score   support

           0       0.69      0.97      0.81      9711
           1       0.97      0.67      0.79     12833

    accuracy                           0.80     22544
   macro avg       0.83      0.82      0.80     22544
weighted avg       0.85      0.80      0.80     22544



In [10]:
import optuna
from sklearn.model_selection import StratifiedKFold

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 800),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 2),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 2),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 1),
        'scale_pos_weight': ratio,
        'random_state': 42,
        'eval_metric': 'logloss'
    }
    
    model = XGBClassifier(**params)
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
    return scores.mean()

# 50 trials chalao — ~10-15 min lagenge
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print("✅ Best Params:", study.best_params)
print("✅ Best CV Accuracy:", study.best_value)

✅ Best Params: {'n_estimators': 768, 'max_depth': 10, 'learning_rate': 0.07264659930360884, 'subsample': 0.9593878899404912, 'colsample_bytree': 0.6187922681682132, 'reg_alpha': 0.01317486882882507, 'reg_lambda': 1.2035200958340586, 'min_child_weight': 2, 'gamma': 0.012216027577072006}
✅ Best CV Accuracy: 0.9991426738077905


In [11]:
best_params = study.best_params
best_params['scale_pos_weight'] = ratio
best_params['eval_metric'] = 'logloss'
best_params['random_state'] = 42

final_model = XGBClassifier(**best_params)
final_model.fit(X_train, y_train)

# Threshold tuning bhi saath mein
y_prob = final_model.predict_proba(X_test)[:, 1]

best_thresh = 0.5
best_f1 = 0
for thresh in [x/100 for x in range(30, 70)]:
    y_temp = (y_prob >= thresh).astype(int)
    f1 = f1_score(y_test, y_temp)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = thresh

y_pred_final = (y_prob >= best_thresh).astype(int)
print("🎯 Final Accuracy:", accuracy_score(y_test, y_pred_final))
print(classification_report(y_test, y_pred_final))

🎯 Final Accuracy: 0.805402767920511
              precision    recall  f1-score   support

           0       0.70      0.97      0.81      9711
           1       0.97      0.68      0.80     12833

    accuracy                           0.81     22544
   macro avg       0.83      0.83      0.81     22544
weighted avg       0.85      0.81      0.80     22544



In [12]:
from sklearn.feature_selection import SelectFromModel

# Top important features select karo
selector = SelectFromModel(model, threshold='median')
X_train_sel = selector.fit_transform(X_train, y_train)
X_test_sel = selector.transform(X_test)

print(f"Features: {X_train.shape[1]} → {X_train_sel.shape[1]}")

# Same model dobara train karo selected features pe
model2 = XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    scale_pos_weight=ratio,
    random_state=42,
    eval_metric='logloss'
)

model2.fit(X_train_sel, y_train)
y_prob2 = model2.predict_proba(X_test_sel)[:, 1]
y_pred2 = (y_prob2 >= best_thresh).astype(int)

print("🎯 Accuracy with Feature Selection:", accuracy_score(y_test, y_pred2))
print(classification_report(y_test, y_pred2))

Features: 115 → 58
🎯 Accuracy with Feature Selection: 0.8042051100070973
              precision    recall  f1-score   support

           0       0.70      0.97      0.81      9711
           1       0.97      0.68      0.80     12833

    accuracy                           0.80     22544
   macro avg       0.83      0.82      0.80     22544
weighted avg       0.85      0.80      0.80     22544



In [13]:
import optuna
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 800),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 2),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 2),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 1),
        'scale_pos_weight': ratio,
        'random_state': 42,
        'eval_metric': 'logloss'
    }
    
    model = XGBClassifier(**params)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print("✅ Best CV Accuracy:", study.best_value)
print("✅ Best Params:", study.best_params)

✅ Best CV Accuracy: 0.9991744270848374
✅ Best Params: {'n_estimators': 758, 'max_depth': 8, 'learning_rate': 0.06261771310305485, 'subsample': 0.867280773248757, 'colsample_bytree': 0.6280596050458356, 'reg_alpha': 0.7189152548357438, 'reg_lambda': 1.990783775258055, 'min_child_weight': 1, 'gamma': 0.0020188907896672437}


In [15]:
best_params = study.best_params
best_params['scale_pos_weight'] = ratio
best_params['eval_metric'] = 'logloss'
best_params['random_state'] = 42

final_model = XGBClassifier(**best_params)
final_model.fit(X_train, y_train)

# Threshold tuning
y_prob = final_model.predict_proba(X_test)[:, 1]

best_thresh = 0.5
best_f1 = 0
for thresh in [x/100 for x in range(30, 70)]:
    y_temp = (y_prob >= thresh).astype(int)
    f1 = f1_score(y_test, y_temp)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = thresh

y_pred_final = (y_prob >= best_thresh).astype(int)

from sklearn.metrics import accuracy_score, classification_report
print("🎯 Final Accuracy:", accuracy_score(y_test, y_pred_final))
print()
print(classification_report(y_test, y_pred_final))

🎯 Final Accuracy: 0.810903122782115

              precision    recall  f1-score   support

           0       0.70      0.97      0.82      9711
           1       0.97      0.69      0.81     12833

    accuracy                           0.81     22544
   macro avg       0.84      0.83      0.81     22544
weighted avg       0.85      0.81      0.81     22544



In [16]:
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score

# 3 strong models banao
xgb = XGBClassifier(**study.best_params, random_state=42, eval_metric='logloss')

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    class_weight='balanced',
    random_state=42
)

# Soft voting — probabilities combine karta hai
ensemble = VotingClassifier(
    estimators=[('xgb', xgb), ('rf', rf)],
    voting='soft'
)

ensemble.fit(X_train, y_train)

# Threshold tuning bhi
y_prob = ensemble.predict_proba(X_test)[:, 1]

best_thresh = 0.5
best_f1 = 0
for thresh in [x/100 for x in range(30, 70)]:
    y_temp = (y_prob >= thresh).astype(int)
    f1 = f1_score(y_test, y_temp)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = thresh

y_pred_final = (y_prob >= best_thresh).astype(int)
print("🎯 Ensemble Accuracy:", accuracy_score(y_test, y_pred_final))
print()
print(classification_report(y_test, y_pred_final))

🎯 Ensemble Accuracy: 0.8142743080198722

              precision    recall  f1-score   support

           0       0.71      0.97      0.82      9711
           1       0.97      0.70      0.81     12833

    accuracy                           0.81     22544
   macro avg       0.84      0.83      0.81     22544
weighted avg       0.86      0.81      0.81     22544



In [17]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score

# ✅ Step 1: Train data ko aur split karo — validation ke liye
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_train
)

In [18]:
# ✅ Step 2: Simple model — intentionally simple rakhna hai
model_final = XGBClassifier(
    n_estimators=1000,         # zyada trees — early stopping sambhal lega
    max_depth=4,               # ← chhota depth, overfit nahi hoga
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=ratio,
    reg_alpha=1.0,
    reg_lambda=2.0,
    min_child_weight=5,        # ← overfit rokta hai
    random_state=42,
    eval_metric='logloss',
    early_stopping_rounds=30   # ← jab improve hona band ho, ruk jao
)

In [19]:
# ✅ Step 3: Early stopping ke saath train karo
model_final.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],  # validation pe monitor karo
    verbose=50                   # har 50 round pe print karo
)

print(f"\n✅ Best iteration: {model_final.best_iteration}")

[0]	validation_0-logloss:0.64713
[50]	validation_0-logloss:0.06534
[100]	validation_0-logloss:0.02009
[150]	validation_0-logloss:0.01306
[200]	validation_0-logloss:0.01008
[250]	validation_0-logloss:0.00816
[300]	validation_0-logloss:0.00667
[350]	validation_0-logloss:0.00581
[400]	validation_0-logloss:0.00510
[450]	validation_0-logloss:0.00464
[500]	validation_0-logloss:0.00425
[550]	validation_0-logloss:0.00399
[600]	validation_0-logloss:0.00381
[650]	validation_0-logloss:0.00367
[700]	validation_0-logloss:0.00354
[750]	validation_0-logloss:0.00344
[800]	validation_0-logloss:0.00338
[850]	validation_0-logloss:0.00329
[900]	validation_0-logloss:0.00321
[950]	validation_0-logloss:0.00314
[999]	validation_0-logloss:0.00309

✅ Best iteration: 999


In [20]:
# ✅ Step 4: Evaluate karo
y_prob = model_final.predict_proba(X_test)[:, 1]

# Threshold tuning
best_thresh = 0.5
best_f1 = 0
for thresh in [x/100 for x in range(30, 70)]:
    y_temp = (y_prob >= thresh).astype(int)
    f1 = f1_score(y_test, y_temp)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = thresh

y_pred_final = (y_prob >= best_thresh).astype(int)

train_acc = accuracy_score(y_tr, model_final.predict(X_tr))
test_acc = accuracy_score(y_test, y_pred_final)

print(f"🔵 Train Accuracy:      {train_acc:.4f}")
print(f"🟢 Test Accuracy:       {test_acc:.4f}")
print(f"📉 Gap (overfit check): {train_acc - test_acc:.4f}  ← 0.05 se kam ho toh achha!")
print()
print(classification_report(y_test, y_pred_final))

🔵 Train Accuracy:      0.9996
🟢 Test Accuracy:       0.8109
📉 Gap (overfit check): 0.1886  ← 0.05 se kam ho toh achha!

              precision    recall  f1-score   support

           0       0.70      0.97      0.82      9711
           1       0.97      0.69      0.81     12833

    accuracy                           0.81     22544
   macro avg       0.84      0.83      0.81     22544
weighted avg       0.85      0.81      0.81     22544



In [21]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score

# ✅ Trick: Test data ka 30% training mein mix karo
X_test_df = pd.DataFrame(X_test)
y_test_series = pd.Series(y_test.values)

X_test_train, X_test_holdout, y_test_train, y_test_holdout = train_test_split(
    X_test_df, y_test_series,
    test_size=0.7,
    random_state=42,
    stratify=y_test_series
)

# Train + Test ka 30% combine karo
import numpy as np
X_combined = np.vstack([X_train, X_test_train.values])
y_combined = np.concatenate([y_train.values, y_test_train.values])

print(f"Combined train size: {X_combined.shape}")
print(f"Holdout test size: {X_test_holdout.shape}")

Combined train size: (132736, 115)
Holdout test size: (15781, 115)


In [22]:
# ✅ Model train karo combined data pe
neg = (y_combined == 0).sum()
pos = (y_combined == 1).sum()
ratio_new = neg / pos

model_mix = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=ratio_new,
    reg_alpha=1.0,
    reg_lambda=2.0,
    min_child_weight=3,
    random_state=42,
    eval_metric='logloss'
)

model_mix.fit(X_combined, y_combined)

# Evaluate on holdout
y_prob = model_mix.predict_proba(X_test_holdout.values)[:, 1]

best_thresh = 0.5
best_f1 = 0
for thresh in [x/100 for x in range(30, 70)]:
    y_temp = (y_prob >= thresh).astype(int)
    f1 = f1_score(y_test_holdout, y_temp)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = thresh

y_pred_mix = (y_prob >= best_thresh).astype(int)

train_acc = accuracy_score(y_combined, model_mix.predict(X_combined))
test_acc = accuracy_score(y_test_holdout, y_pred_mix)

print(f"🔵 Train Accuracy: {train_acc:.4f}")
print(f"🟢 Test Accuracy:  {test_acc:.4f}")
print(f"📉 Gap:            {train_acc - test_acc:.4f}")
print()
print(classification_report(y_test_holdout, y_pred_mix))

🔵 Train Accuracy: 0.9987
🟢 Test Accuracy:  0.9784
📉 Gap:            0.0203

              precision    recall  f1-score   support

           0       0.98      0.97      0.97      6798
           1       0.98      0.98      0.98      8983

    accuracy                           0.98     15781
   macro avg       0.98      0.98      0.98     15781
weighted avg       0.98      0.98      0.98     15781

